In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.io import MemoryFile
from rasterio.transform import from_bounds
from rasterio.warp import (
    calculate_default_transform,
    reproject
)
from rasterio.windows import from_bounds as window_from_bounds

In [ ]:
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_DEPMAP_ORG = DIR_DATA / "1_org" / "3_Terrain" / "depmap"
DIR_DEPMAP_PROCESSED = DIR_DATA / "2_processed" / "3_Terrain" / "depmap"

FILEPATH_PATCH_CENTERS = DIR_METADATA / "patch-centers.csv"
FILEPATH_DEPMAP = DIR_DEPMAP_ORG / "depression_mslli_min5_max4505_stp5_lin100.tif"

TARGET_CRS = "EPSG:2958"
SAR_RESOLUTION = 0.5
PATCH_SIZE = 256
GROUND_SIZE = PATCH_SIZE * SAR_RESOLUTION  # 128 m
HALF_SIZE = GROUND_SIZE / 2  # 64 m

In [ ]:
patches = pd.read_csv(FILEPATH_PATCH_CENTERS)
# Remove empty rows
patches = patches.dropna(subset=["id"])

with rasterio.open(FILEPATH_DEPMAP) as src:
    print("====================================")
    print("Original terrain raster")
    print(f"CRS        : {src.crs}")
    print(f"Resolution : {src.res}")
    print("====================================")

    # --------------------------------------------------------
    # Reproject only if necessary
    # --------------------------------------------------------
    if src.crs.to_string() != TARGET_CRS:
        transform, width, height = calculate_default_transform(
            src.crs,
            TARGET_CRS,
            src.width,
            src.height,
            *src.bounds
        )
        profile = src.profile.copy()
        profile.update(
            crs=TARGET_CRS,
            transform=transform,
            width=width,
            height=height,
            compress="LZW"
        )
        memfile = MemoryFile()
        with memfile.open(**profile) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                resampling=Resampling.bilinear
            )
        terrain_src = memfile.open()
    else:
        terrain_src = src

    print("====================================")
    print("Working terrain raster")
    print(f"CRS        : {terrain_src.crs}")
    print(f"Resolution : {terrain_src.res}")
    print("====================================")

    # ========================================================
    # Extract patches
    # ========================================================

    for _, patch in patches.iterrows():
        patch_id = patch["id"]
        patch_year = int(patch["year"])

        output_dir = DIR_DEPMAP_PROCESSED / f"patches_{patch_year}"
        output_dir.mkdir(parents=True, exist_ok=True)

        x = patch["x"]
        y = patch["y"]

        xmin = x - HALF_SIZE
        xmax = x + HALF_SIZE
        ymin = y - HALF_SIZE
        ymax = y + HALF_SIZE

        window = window_from_bounds(
            xmin,
            ymin,
            xmax,
            ymax,
            transform=terrain_src.transform
        )
        window = window.round_offsets().round_lengths()

        if (
            window.col_off < 0
            or window.row_off < 0
            or window.col_off + window.width > terrain_src.width
            or window.row_off + window.height > terrain_src.height
        ):
            print(f"{patch_id} is out of bounds.")
            continue

        patch_img = terrain_src.read(
            1,
            window=window,
            out_shape=(PATCH_SIZE, PATCH_SIZE),
            resampling=Resampling.bilinear
        ).astype(np.float32)

        profile = terrain_src.profile.copy()

        profile.update(
            driver="GTiff",
            compress="LZW",
            dtype="float32",
            count=1,
            width=PATCH_SIZE,
            height=PATCH_SIZE,
            transform=from_bounds(
                xmin,
                ymin,
                xmax,
                ymax,
                PATCH_SIZE,
                PATCH_SIZE
            )
        )

        output_path = output_dir / f"depmap_{patch_id}.tif"

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(patch_img, 1)

print("Done.")